# statphys-viz: Temperature Scan Demo

This notebook demonstrates automated temperature scanning of the 2D Ising model
with quantitative validation against the Onsager exact solution.

In [ ]:
# Install if needed
# %pip install -e .[web]

import matplotlib.pyplot as plt
import numpy as np

from statphys_viz import IsingConfig, IsingModel, OnsagerBenchmark
from statphys_viz.visualize import ONSAGER_TC, plot_observables

print(f"Onsager Tc = {ONSAGER_TC:.6f}")

## Single Temperature Measurement

In [ ]:
# Configure near critical temperature
config = IsingConfig(L=64, T=2.269, J=1.0, algorithm="wolff", seed=42)
model = IsingModel(config)

# Thermalize
model.thermalize(500)

# Measure
results = model.measure(2000)

print(f"Energy/spin:       {results['energy_mean']:.6f}")
print(f"|M|/spin:          {results['magnetization_mean']:.6f}")
print(f"Susceptibility:    {results['susceptibility']:.2f}")
print(f"Specific heat:     {results['specific_heat']:.4f}")

# Validate against Onsager exact
benchmark = OnsagerBenchmark(L=64)
errors = benchmark.validate(results, T=2.269)
print(f"\nMax relative error: {errors['max_rel_error']:.2%}")
for k, v in errors.items():
    print(f"  {k}: {v:.4f}")

## Temperature Scan with Exact Comparison

In [ ]:
# Scan temperatures across the phase transition
L = 32
T_min, T_max = 1.5, 3.5
n_points = 25
n_samples = 1000
thermalize = 500

temps = np.linspace(T_min, T_max, n_points)
results = {
    "energy_mean": [],
    "magnetization_mean": [],
    "specific_heat": [],
    "susceptibility": [],
}

benchmark = OnsagerBenchmark(L=L)

for i, T in enumerate(temps):
    config = IsingConfig(L=L, T=T, algorithm="wolff")
    model = IsingModel(config)
    model.thermalize(thermalize)
    meas = model.measure(n_samples)

    results["energy_mean"].append(meas["energy_mean"])
    results["magnetization_mean"].append(meas["magnetization_mean"])
    results["specific_heat"].append(meas["specific_heat"])
    results["susceptibility"].append(meas["susceptibility"])

    if (i + 1) % 5 == 0:
        print(f"  T={T:.3f}: E={meas['energy_mean']:.4f}, |M|={meas['magnetization_mean']:.4f}")

# Convert to arrays
for k in results:
    results[k] = np.array(results[k])

In [ ]:
# Plot with exact comparison
fig = plot_observables(results, temps, benchmark.exact_values)
plt.show()

## Print Error Table

In [ ]:
print("\nRelative Errors vs Onsager Exact:")
print(f"{'T':>6}  {'E_rel':>8}  {'M_rel':>8}  {'C_rel':>8}  {'max':>8}")
print("-" * 45)

for i, T in enumerate(temps):
    mc_res = {k: v[i] for k, v in results.items()}
    errors = benchmark.validate(mc_res, T)
    e_err = errors.get("energy_rel_error", 0)
    m_err = errors.get("magnetization_rel_error", 0)
    c_err = errors.get("specific_heat_rel_error", 0)
    max_err = errors["max_rel_error"]
    print(f"{T:6.3f}  {e_err:8.2%}  {m_err:8.2%}  {c_err:8.2%}  {max_err:8.2%}")

## Finite-Size Scaling at Tc

In [ ]:
# Measure magnetization at Tc for different L
Ls = [8, 16, 32, 64, 128]
mags_at_tc = []
chis_at_tc = []

for L in Ls:
    config = IsingConfig(L=L, T=ONSAGER_TC, algorithm="wolff")
    model = IsingModel(config)
    model.thermalize(200)
    results = model.measure(1000)
    mags_at_tc.append(results["magnetization_mean"])
    chis_at_tc.append(results["susceptibility"])
    print(f"L={L:3d}: |M|={results['magnetization_mean']:.4f}, χ={results['susceptibility']:.1f}")

import numpy as np

from statphys_viz.benchmark import fit_critical_exponents

# Fit β/ν from m ~ L^(-β/ν)
slope_m, err_m = fit_critical_exponents(np.array(Ls), np.array(mags_at_tc), "magnetization")
print(f"\nβ/ν = {slope_m:.4f} ± {err_m:.4f} (exact: {-1 / 8:.4f})")

# Fit γ/ν from χ ~ L^(γ/ν)
slope_chi, err_chi = fit_critical_exponents(np.array(Ls), np.array(chis_at_tc), "susceptibility")
print(f"γ/ν = {slope_chi:.4f} ± {err_chi:.4f} (exact: {7 / 4:.4f})")

## Interactive Widget Demo

Run the cell below to launch the interactive Ising widget with live spin lattice visualization.

In [ ]:
# Uncomment to run interactive widget (requires ipywidgets + ipycanvas)
# from statphys_viz.widgets import IsingWidget
# widget = IsingWidget(L=64, T=2.269, algorithm="wolff")
# widget.display()